In [34]:
import glob
import os

import face_recognition


def extract_lfw_encodings(lfw_base_dir, max_images=3000):
    """
    Extracts face encodings from the LFW dataset directory structure.
    Ensures that only one image per person is sampled.

    Args:
        lfw_base_dir (str): Path to the main 'lfw' directory containing the name subfolders.
        max_images (int): Number of images to process for the density check experiment.
    """
    # Navigates the lfw/name/name_xxxx.jpg structure
    search_pattern = os.path.join(lfw_base_dir, '**', '*.jpg')
    image_paths = glob.glob(search_pattern, recursive=True)

    # Shuffle to get a random subset of individuals
    # np.random.shuffle(image_paths)

    encodings = []
    processed_count = 0

    # Initialize a set to keep track of individuals we've already processed
    seen_persons = set()

    print(f"Found {len(image_paths)} total images. Starting extraction for {max_images} images...")

    for img_path in image_paths:
        if processed_count >= max_images:
            break

        # Extract the person's name from the directory name (e.g., 'George_W_Bush')
        person_name = os.path.basename(os.path.dirname(img_path))

        # Check if we already have an encoding for this person
        if person_name in seen_persons:
            continue

        print(f"Processing image {img_path} for {person_name}...")
        try:
            # Load the 250x250 .jpg image
            image = face_recognition.load_image_file(img_path)

            # The face is already centered and cropped, but the model still needs to locate it
            # and map the features into a 128-dimensional encoding
            face_enc = face_recognition.face_encodings(image)

            if len(face_enc) > 0:
                # Append the first detected face's encoding
                encodings.append(face_enc[0])

                # Add the person to our tracking set
                seen_persons.add(person_name)
                processed_count += 1

                if processed_count % 500 == 0:
                    print(f"Processed {processed_count}/{max_images} unique images...")

        except Exception as e:
            print(f"Error processing {img_path}: {e}")
            continue

    encodings_array = np.array(encodings)
    print(f"Extraction complete. Array shape: {encodings_array.shape}")

    # Save the embeddings to a file so they can be loaded for the k-d tree density check
    output_filename = f"lfw_encodings_{max_images}.npy"
    np.save(output_filename, encodings_array)
    print(f"Saved to {output_filename}")

    return encodings_array

In [41]:
dataset_path = "datasets/lfw/lfw-deepfunneled/lfw-deepfunneled"
embeddings = extract_lfw_encodings(dataset_path, max_images=100)

Found 13233 total images. Starting extraction for 100 images...
Processing image datasets/lfw/lfw-deepfunneled/lfw-deepfunneled/Elena_de_Chavez/Elena_de_Chavez_0001.jpg for Elena_de_Chavez...
Processing image datasets/lfw/lfw-deepfunneled/lfw-deepfunneled/Ilan_Goldfajn/Ilan_Goldfajn_0001.jpg for Ilan_Goldfajn...
Processing image datasets/lfw/lfw-deepfunneled/lfw-deepfunneled/Junko_Tabei/Junko_Tabei_0001.jpg for Junko_Tabei...
Processing image datasets/lfw/lfw-deepfunneled/lfw-deepfunneled/David_Welch/David_Welch_0001.jpg for David_Welch...
Processing image datasets/lfw/lfw-deepfunneled/lfw-deepfunneled/Atiabet_Ijan_Amabel/Atiabet_Ijan_Amabel_0001.jpg for Atiabet_Ijan_Amabel...
Processing image datasets/lfw/lfw-deepfunneled/lfw-deepfunneled/Paul_Krueger/Paul_Krueger_0001.jpg for Paul_Krueger...
Processing image datasets/lfw/lfw-deepfunneled/lfw-deepfunneled/Namuddu_Florence/Namuddu_Florence_0001.jpg for Namuddu_Florence...
Processing image datasets/lfw/lfw-deepfunneled/lfw-deepfunneled/

In [36]:
import numpy as np
from scipy.spatial import cKDTree
import time

def compute_density_parameter_C(encodings_file, radius_R):
    """
    Computes the density parameter C using a k-d tree, as described in the paper.

    Args:
        encodings_file (str): Path to the saved numpy array of face encodings.
        radius_R (float): The distance parameter (e.g., 0.8 or 1.0).

    Returns:
        int: The computed density parameter C.
    """
    print(f"Loading encodings from {encodings_file}...")
    encodings = np.load(encodings_file)
    num_points, dim = encodings.shape
    print(f"Loaded {num_points} points of dimension {dim}.")

    print("Building k-d tree...")
    start_time = time.time()

    # 1. Build the k-d tree (matches the paper's preprocessing step)
    tree = cKDTree(encodings)

    # 2. Query the tree for all points within radius R
    # This returns a list of lists, where each inner list contains the indices
    # of neighbors for that specific point.
    print(f"Querying overlaps for Radius R = {radius_R}...")
    neighbors = tree.query_ball_tree(tree, r=radius_R)

    # 3. Calculate degrees (number of neighbors excluding the point itself)
    # The paper defines C = max_degree + 1
    max_degree = 0
    for i, neighbor_list in enumerate(neighbors):
        # Subtract 1 because query_ball_tree includes the point itself (distance 0)
        degree = len(neighbor_list) - 1
        if degree > max_degree:
            max_degree = degree

    # As per the paper's transformation algorithm: C = max deg(v) + 1
    C = max_degree + 1

    end_time = time.time()

    print("-" * 30)
    print(f"Results for R = {radius_R}:")
    print(f"Maximum overlap (max degree) : {max_degree}")
    print(f"Density parameter C          : {C}")
    print(f"Computation time             : {end_time - start_time:.4f} seconds")
    print("-" * 30)

    return C

In [44]:
encodings_path = "lfw_encodings_100.npy"
print("Testing against parameters from Table 1...")
compute_density_parameter_C(encodings_path, radius_R=0.1)
compute_density_parameter_C(encodings_path, radius_R=1.0)

Testing against parameters from Table 1...
Loading encodings from lfw_encodings_100.npy...
Loaded 100 points of dimension 128.
Building k-d tree...
Querying overlaps for Radius R = 0.1...
------------------------------
Results for R = 0.1:
Maximum overlap (max degree) : 0
Density parameter C          : 1
Computation time             : 0.0020 seconds
------------------------------
Loading encodings from lfw_encodings_100.npy...
Loaded 100 points of dimension 128.
Building k-d tree...
Querying overlaps for Radius R = 1.0...
------------------------------
Results for R = 1.0:
Maximum overlap (max degree) : 99
Density parameter C          : 100
Computation time             : 0.0027 seconds
------------------------------


100

In [15]:
lfw_base_dir = "datasets/lfw/lfw-deepfunneled/lfw-deepfunneled/George_W_Bush"
# get the encodings of a George_W_Bush
search_pattern = os.path.join(lfw_base_dir, '**', '*.jpg')
image_paths = glob.glob(search_pattern, recursive=True)

encodings = []
processed_count = 0

print(f"Found {len(image_paths)} total images. Starting extraction for George_W_Bush images...")

for img_path in image_paths:
    print(f"Processing image {img_path}...")
    try:
        # Load the 250x250 .jpg image
        image = face_recognition.load_image_file(img_path)

        # The face is already centered and cropped, but the model still needs to locate it
        # and map the features into a 128-dimensional encoding
        face_enc = face_recognition.face_encodings(image)

        if len(face_enc) > 0:
            # Append the first detected face's encoding
            encodings.append(face_enc[0])
            processed_count += 1

            if processed_count % 500 == 0:
                print(f"Processed {processed_count} images...")

    except Exception as e:
        print(f"Error processing {img_path}: {e}")
        continue

encodings_array = np.array(encodings)
print(f"Extraction complete. Array shape: {encodings_array.shape}")

# Save the embeddings to a file so they can be loaded for the k-d tree density check
output_filename = f"lfw_encodings_Bush.npy"
np.save(output_filename, encodings_array)
print(f"Saved to {output_filename}")

Found 530 total images. Starting extraction for George_W_Bush images...
Processing image datasets/lfw/lfw-deepfunneled/lfw-deepfunneled/George_W_Bush/George_W_Bush_0077.jpg...
Processing image datasets/lfw/lfw-deepfunneled/lfw-deepfunneled/George_W_Bush/George_W_Bush_0424.jpg...
Processing image datasets/lfw/lfw-deepfunneled/lfw-deepfunneled/George_W_Bush/George_W_Bush_0386.jpg...
Processing image datasets/lfw/lfw-deepfunneled/lfw-deepfunneled/George_W_Bush/George_W_Bush_0493.jpg...
Processing image datasets/lfw/lfw-deepfunneled/lfw-deepfunneled/George_W_Bush/George_W_Bush_0207.jpg...
Processing image datasets/lfw/lfw-deepfunneled/lfw-deepfunneled/George_W_Bush/George_W_Bush_0177.jpg...
Processing image datasets/lfw/lfw-deepfunneled/lfw-deepfunneled/George_W_Bush/George_W_Bush_0036.jpg...
Processing image datasets/lfw/lfw-deepfunneled/lfw-deepfunneled/George_W_Bush/George_W_Bush_0401.jpg...
Processing image datasets/lfw/lfw-deepfunneled/lfw-deepfunneled/George_W_Bush/George_W_Bush_0519

In [ ]:
encodings_path = "lfw_encodings_Bush.npy"
compute_density_parameter_C(encodings_path, radius_R=0.65)
compute_density_parameter_C(encodings_path, radius_R=1.0)

In [ ]:
# note: to run this, we need `pip install "numpy<2.0.0"`
import os
import glob
import numpy as np
from deepface import DeepFace

def extract_facenet_encodings(lfw_base_dir, max_images=3000):
    """
    Extracts FaceNet encodings from the LFW dataset.

    Args:
        lfw_base_dir (str): Path to the extracted 'lfw' directory.
        max_images (int): Number of images to process (e.g., 3000).

    Returns:
        numpy.ndarray: An array of FaceNet embeddings.
    """
    # LFW images are stored as lfw/name/name_xxxx.jpg
    search_pattern = os.path.join(lfw_base_dir, '**', '*.jpg')
    image_paths = glob.glob(search_pattern, recursive=True)

    # Shuffle to get a random subset
    np.random.shuffle(image_paths)

    encodings = []
    processed_count = 0

    print(f"Found {len(image_paths)} total images.")
    print(f"Starting FaceNet extraction for {max_images} images...")

    for img_path in image_paths:
        if processed_count >= max_images:
            break

        try:
            # enforce_detection=False prevents crashes if the detector misses a heavily obscured face
            # model_name="Facenet" loads the specific weights that align with R=0.8 / 1.0
            representation = DeepFace.represent(
                img_path=img_path,
                model_name="Facenet",
                enforce_detection=False
            )

            # DeepFace returns a list of faces found in the image.
            # LFW has one centered face per image.
            if len(representation) > 0:
                embedding = representation[0]["embedding"]
                encodings.append(embedding)
                processed_count += 1

                if processed_count % 100 == 0:
                    print(f"Processed {processed_count}/{max_images} images...")

        except Exception as e:
            print(f"Error processing {img_path}: {e}")
            continue

    encodings_array = np.array(encodings)
    print(f"Extraction complete. Array shape: {encodings_array.shape}")

    # Save the embeddings to a file
    output_filename = f"lfw_facenet_encodings_{max_images}.npy"
    np.save(output_filename, encodings_array)
    print(f"Saved to {output_filename}")

    return encodings_array

In [ ]:
dataset_path = "datasets/lfw/lfw-deepfunneled/lfw-deepfunneled"
embeddings = extract_facenet_encodings(dataset_path, max_images=3000)

In [39]:
import numpy as np

def verify_globally_disjoint(encodings, R, delta):
    """
    Verifies if a dataset is globally disjoint after rotation.

    Parameters:
    encodings (np.ndarray): The dataset of shape (num_points, dim).
    R (np.ndarray): The rotation matrix of shape (dim, dim).
    delta (float): The minimum distance threshold.

    Returns:
    is_disjoint (bool): True if the condition holds for all dimensions.
    global_min_gap (float): The absolute smallest gap found across all dimensions.
    """

    # 1. Apply the rotation
    # If R is designed to multiply column vectors (R * x), then for a matrix
    # of row vectors X, the rotated matrix is X @ R.T
    rotated_encodings = encodings @ R.T

    # 2. Sort the projections along each dimension independently
    # axis=0 sorts each column (dimension) from smallest to largest
    sorted_encodings = np.sort(rotated_encodings, axis=0)

    # 3. Calculate the distance between adjacent points in the sorted arrays
    # np.diff subtracts each element from the next one along the specified axis
    gaps = np.diff(sorted_encodings, axis=0)

    # 4. Find the minimum gap
    # First find the minimum gap for each of the 128 dimensions
    min_gap_per_axis = np.min(gaps, axis=0)

    # Then find the absolute smallest gap across all dimensions
    global_min_gap = np.min(min_gap_per_axis)

    # 5. Check against the threshold
    is_disjoint = global_min_gap > delta

    return is_disjoint, global_min_gap, min_gap_per_axis

In [46]:
encodings_file = 'lfw_encodings_100.npy'
print(f"Loading encodings from {encodings_file}...")
encodings = np.load(encodings_file)
num_points, dim = encodings.shape
print(f"Loaded {num_points} points of dimension {dim}.")

delta = 0.1
R_test = np.eye(dim)

is_disjoint, min_gap, min_gaps_array = verify_globally_disjoint(encodings, R_test, delta)

print(f"Globally Disjoint: {is_disjoint}")
print(f"Smallest gap found: {min_gap:.6f} (Needs to be > {delta})")

Loading encodings from lfw_encodings_100.npy...
Loaded 100 points of dimension 128.
Globally Disjoint: False
Smallest gap found: 0.000001 (Needs to be > 0.1)


In [47]:
import numpy as np
from scipy.spatial import cKDTree

def minimal_length_in_dataset(encodings_array):
    """
    Finds the minimum distance between ANY two distinct points in a large dataset.
    Uses a k-d tree for O(N log N) time complexity instead of O(N^2).
    """
    print("Building k-d tree...")
    tree = cKDTree(encodings_array)

    # Query the 2 nearest neighbors for every single point.
    # k=1 is the point itself (distance 0).
    # k=2 is the closest DIFFERENT point.
    print("Querying nearest neighbors...")
    distances, indices = tree.query(encodings_array, k=2)

    # distances[:, 1] extracts the distance to the 2nd nearest neighbor
    # (the closest distinct point) for every item in the dataset
    nearest_distinct_distances = distances[:, 1]

    # Find the absolute minimum distance in that list
    min_dist = np.min(nearest_distinct_distances)

    # (Optional) Find exactly which two points those belong to
    point_a_index = np.argmin(nearest_distinct_distances)
    point_b_index = indices[point_a_index, 1]

    print(f"The minimum distance is between index {point_a_index} and {point_b_index}")

    return min_dist

In [48]:
encodings_path = "lfw_encodings_100.npy"
embeddings = np.load(encodings_path)
min_dist = minimal_length_in_dataset(embeddings)
print(f"Minimal length between any two different points: {min_dist}")

Building k-d tree...
Querying nearest neighbors...
The minimum distance is between index 2 and 44
Minimal length between any two different points: 0.46183297633148934
